# Lekce 04 - Větvení

Obsah:

* proč větvení
* `if`, blok a odsazení
* `else` a `elif`, na pořadí podmínek záleží
* složené podmínky: `and`, `or`, `not`, řetězené porovnání
* vnořené podmínky a jak se jim vyhnout
* pravdivostní hodnota (truthiness), `is None`
* podmíněný výraz `a if podminka else b`
* podmínky nad kolekcemi: `in`, `min`, `max`
* `match` / `case`
* kontrola vstupu
* časté chyby
* výraz a příkaz

Cykly zatím nepotřebujeme. Každou podmínku vyhodnotíme jednou, nad jednou hodnotou
nebo nad celou kolekcí najednou. Cykly přijdou v lekci 5.

## 1. Proč větvení

Dosud program prováděl vždy všechny řádky odshora dolů. V lekci 2 jsme uměli
zjistit, **jestli** prvek vyhovuje - výsledkem bylo `True` nebo `False`. Teď
podle toho budeme **jednat**: vypsat jiné hlášení, spočítat jinou hodnotu,
zvolit jiný postup.

In [41]:
M_Ed = 82.0     # kNm, navrhovy ohybovy moment
M_Rd = 95.0     # kNm, unosnost prurezu

print(M_Ed <= M_Rd)    # z lekce 2: jen hodnota porovnani

True


## 2. `if`, blok a odsazení

```python
if podminka:
    prikaz_1
    prikaz_2
dalsi_prikaz
```

* za podmínkou je **dvojtečka**
* příkazy, které se mají provést jen při splnění podmínky, tvoří **blok**
* blok se pozná podle **odsazení**, standardně 4 mezery (VS Code je vloží
  klávesou Tab)
* první neodsazený řádek už do bloku nepatří a provede se vždy

Python nemá složené závorky ani `end` - odsazení je součást syntaxe.

In [42]:
M_Ed = 82.0
M_Rd = 95.0
vyuziti = M_Ed / M_Rd

if vyuziti <= 1.0:
    print("Prurez vyhovuje")
    print(f"Rezerva {1 - vyuziti:.0%}")

print(f"Vyuziti {vyuziti:.2f}")    # provede se vzdy

Prurez vyhovuje
Rezerva 14%
Vyuziti 0.86


## 3. `else` a `elif`

`else` je blok pro případ, kdy podmínka splněná není. Provede se právě jeden
z obou bloků.

In [43]:
M_Ed = 104.0 # zatizeni, effect
M_Rd = 95.0 # unosnost, resistance
vyuziti = M_Ed / M_Rd

if vyuziti <= 1.0:
    print(f"VYHOVUJE (vyuziti {vyuziti:.2f})")
else:
    print(f"NEVYHOVUJE (vyuziti {vyuziti:.2f})")

NEVYHOVUJE (vyuziti 1.09)


Pro víc než dvě možnosti slouží `elif` (zkratka z *else if*). Podmínky se
testují **odshora** a provede se blok **první** splněné podmínky. Ostatní se
přeskočí, i kdyby byly také splněné.

Příklad: zatřídění součinitele prostupu tepla obvodové stěny podle
ČSN 73 0540-2 (zjednodušeně).

In [44]:
U = 0.22     # W/(m2.K)

if U <= 0.18:
    hodnoceni = "doporucena pro pasivni domy"
elif U <= 0.25:
    hodnoceni = "doporucena"
elif U <= 0.30:
    hodnoceni = "pozadovana"
else:
    hodnoceni = "nevyhovuje"

print(f"U = {U} W/(m2.K): {hodnoceni}")

U = 0.22 W/(m2.K): doporucena


### Na pořadí záleží

Stejné podmínky v opačném pořadí dají špatný výsledek. `U = 0.15` splní hned
první podmínku `U <= 0.30` a k přísnějším se program vůbec nedostane.

In [45]:
U = 0.15

if U <= 0.30:
    hodnoceni = "pozadovana"
elif U <= 0.25:
    hodnoceni = "doporucena"
elif U <= 0.18:
    hodnoceni = "doporucena pro pasivni domy"
else:
    hodnoceni = "nevyhovuje"

print(f"U = {U} W/(m2.K): {hodnoceni}")    # spatne!

U = 0.15 W/(m2.K): pozadovana


Pravidlo: u řady mezí `<=` začněte **nejpřísnější** (nejmenší) mezí, u řady
mezí `>=` naopak největší.

`else` na konci není povinné. Pokud chybí a žádná podmínka neplatí, neprovede
se nic.

## 4. Složené podmínky

Z lekce 2 známe `and`, `or`, `not`. V podmínce fungují stejně:

| Operátor | Výsledek je `True`, když |
|---|---|
| `a and b` | platí obě |
| `a or b` | platí aspoň jedna |
| `not a` | `a` neplatí |

Přednost má `not`, potom `and`, nakonec `or`. Když si nejste jistí,
použijte závorky - čtenáři pomohou vždy.

In [46]:
beton = "C30/37"
kryti = 30.0          # mm
trida_prostredi = "XC4"

if beton == "C30/37" and kryti >= 30.0:
    print("Beton i kryti odpovidaji projektu")

if trida_prostredi == "XC4" or trida_prostredi == "XD1":
    print("Konstrukce je vystavena vlhku a stridani vlhkosti")

if not kryti < 25.0:
    print("Kryti neni mensi nez 25 mm")

Beton i kryti odpovidaji projektu
Konstrukce je vystavena vlhku a stridani vlhkosti
Kryti neni mensi nez 25 mm


Interval se v Pythonu zapisuje **řetězeným porovnáním**, stejně jako v matematice:

In [47]:
x = 3.4     # m, poloha bodu na nosniku
L = 6.0     # m, rozpeti

if 0 <= x <= L: # Zkratka pro (0 <= x) and (x <= L), nikoliv (0 <= x) <= L
    print("Bod lezi na nosniku")
else:
    print("Bod je mimo nosnik")

Bod lezi na nosniku


### Zkrácené vyhodnocení

`and` a `or` vyhodnocují zleva a skončí, jakmile je výsledek jasný. Když je
vlevo od `and` hodnota `False`, pravá strana se vůbec nespočítá. Toho se
využívá k ochraně před chybou:

In [48]:
plocha = 0.0       # m2, plocha dosud nezadana
sila = 120.0       # kN

if plocha > 0 and sila / plocha > 5000:   # deleni se pri plocha = 0 neprovede
    print("Prekroceno napeti")
else:
    print("Napeti nelze spocitat nebo je v poradku")

Napeti nelze spocitat nebo je v poradku


## 5. Vnořené podmínky

Blok `if` může obsahovat další `if`. Dvě až tři úrovně se ještě dají číst,
víc už ne.

In [49]:
rozpeti = 6.0     # m
pruhyb = 21.0     # mm
M_Ed = 82.0 # zatizeni
M_Rd = 95.0 # unosnost

if M_Ed <= M_Rd:
    if pruhyb <= rozpeti * 1000 / 250:
        print("Vyhovuje unosnost i pruhyb")
    else:
        print("Vyhovuje unosnost, nevyhovuje pruhyb")
else:
    print("Nevyhovuje unosnost")

Vyhovuje unosnost i pruhyb


Často jde vnoření nahradit plochou řadou `elif`, nebo si podmínky předem uložit
do pojmenovaných proměnných. Kód pak čte skoro jako věta:

In [50]:
unosnost_ok = M_Ed <= M_Rd
pruhyb_ok = pruhyb <= rozpeti * 1000 / 250

if unosnost_ok and pruhyb_ok:
    print("Vyhovuje unosnost i pruhyb")
elif unosnost_ok:
    print("Vyhovuje unosnost, nevyhovuje pruhyb")
else:
    print("Nevyhovuje unosnost")

Vyhovuje unosnost i pruhyb


## 6. Pravdivostní hodnota (truthiness)

Za `if` nemusí stát jen `True` nebo `False`. Python umí rozhodnout o pravdivosti
**libovolné** hodnoty. Za nepravdivé se považují:

* `False` a `None`
* nula: `0`, `0.0`
* prázdné kolekce a řetězce: `""`, `[]`, `()`, `{}`, `set()`

Všechno ostatní je pravdivé. Ověřit to jde funkcí `bool()`:

In [51]:
print(bool(0), bool(0.0), bool(""), bool([]), bool({}), bool(None))
print(bool(12), bool(-0.5), bool("C25/30"), bool([0]), bool(" "))

False False False False False False
True True True True True


In [52]:
poznamka = ""
vady = ["trhlina v rohu"]

if poznamka:
    print("Poznamka:", poznamka)
else:
    print("Bez poznamky")

if vady:
    print(f"Pocet zaznamenanych vad: {len(vady)}")

Bez poznamky
Pocet zaznamenanych vad: 1


### Pozor na nulu a `None`

Chybějící hodnota (`None`) a hodnota nula jsou dvě různé věci. Tloušťka izolace
`0.0` je platný údaj (izolace není), `None` znamená, že ji nikdo nezadal. Test
`if tloustka:` je od sebe nerozliší. Na chybějící hodnotu se ptejte přímo pomocí
`is None`:

In [53]:
tloustka = 0.0     # izolace neni - platny udaj

if not tloustka:
    print("Spatne: tvari se to jako chybejici udaj")

if tloustka is None:
    print("Tloustka nebyla zadana")
else:
    print(f"Tloustka izolace: {tloustka} m")

Spatne: tvari se to jako chybejici udaj
Tloustka izolace: 0.0 m


## 7. Podmíněný výraz

Když se podle podmínky volí jen **hodnota**, stačí jeden řádek:

```python
vysledek = hodnota_kdyz_ano if podminka else hodnota_kdyz_ne
```

Hodí se na krátké volby, typicky do f-stringu. Na nic složitějšího ho
nepoužívejte - čitelnost má přednost.

In [54]:
vyuziti = 0.86

stav = "VYHOVUJE" if vyuziti <= 1.0 else "NEVYHOVUJE"
print(f"Vyuziti {vyuziti:.2f} - {stav}")

pocet = 1
print(f"{pocet} {'prvek' if pocet == 1 else 'prvky'}")

Vyuziti 0.86 - VYHOVUJE
1 prvek


## 8. Podmínky nad kolekcemi

I bez cyklu se dá rozhodovat o celé kolekci najednou. Pomohou funkce z lekce 3:

* `in` - je hodnota v seznamu, množině, mezi klíči slovníku?
* `max(vyuziti) <= 1` - vyhovují **všechny** prvky? (stačí, když vyhovuje ten nejhorší)
* `min(vyuziti) > 1` - nevyhovuje **žádný** prvek?
* `len(seznam) == 0` nebo jen `not seznam` - je seznam prázdný?

In [55]:
vyuziti = [0.72, 0.91, 1.04, 0.65]

if max(vyuziti) <= 1.0:
    print("Vsechny prvky vyhovuji")
else:
    nejhorsi = vyuziti.index(max(vyuziti))
    print(f"Nevyhovuje prvek s indexem {nejhorsi} (vyuziti {max(vyuziti):.2f})")

Nevyhovuje prvek s indexem 2 (vyuziti 1.04)


In [56]:
prvek = {"kod": "SO-01_2NP", "material": "C25/30", "objem": 12.8}

if "vyztuz" in prvek:
    print("Vyztuz:", prvek["vyztuz"])
else:
    print("Atribut vyztuz chybi")

povolene_betony = {"C25/30", "C30/37"}
if prvek["material"] not in povolene_betony:
    print("Material neni na seznamu povolenych")
else:
    print("Material je povoleny")

Atribut vyztuz chybi
Material je povoleny


## 9. `match` / `case`

Od Pythonu verze 3.10 existuje příkaz `match`. Porovná jednu hodnotu s řadou **vzorů**
a provede blok prvního vzoru, který sedí. Pro výběr z pevného seznamu možností
je přehlednější než dlouhá řada `elif ... == ...`.

```python
match hodnota:
    case vzor_1:
        ...
    case vzor_2 | vzor_3:     # vice moznosti
        ...
    case _:                   # cokoli jineho (jako else)
        ...
```

Příklad: minimální krytí výztuže z hlediska trvanlivosti podle třídy prostředí
(EN 1992-1-1, konstrukční třída S4).

In [57]:
trida_prostredi = "XC3"

match trida_prostredi:
    case "X0":
        c_min_dur = 10
    case "XC1":
        c_min_dur = 15
    case "XC2" | "XC3":
        c_min_dur = 25
    case "XC4":
        c_min_dur = 30
    case "XD1" | "XS1":
        c_min_dur = 35
    case _:
        c_min_dur = None

if c_min_dur is None:
    print(f"Neznama trida prostredi {trida_prostredi}")
else:
    print(f"{trida_prostredi}: c_min,dur = {c_min_dur} mm")

XC3: c_min,dur = 25 mm


Stejný výsledek by šel zapsat řadou `elif`. Kdy je `match` lepší:

* porovnáváme **jednu** hodnotu s řadou konstant
* chceme sloučit několik možností operátorem `|`

Kdy zůstat u `if` / `elif`: podmínky jsou nerovnosti (`U <= 0.25`) nebo
se týkají různých proměnných.

### Vzory s rozbalením

`match` umí víc než porovnání s konstantou. Vzor může mít **tvar** - třeba
tuple o dvou prvcích - a jeho části se rovnou uloží do proměnných. Za vzor lze
připsat i podmínku `if` (tzv. guard).

In [58]:
bod = (0.0, 4.5)

match bod:
    case (0.0, 0.0):
        print("Pocatek souradnic")
    case (0.0, y):
        print(f"Bod na ose y, y = {y}")
    case (x, 0.0):
        print(f"Bod na ose x, x = {x}")
    case (x, y) if x > 0 and y > 0:
        print(f"Bod v 1. kvadrantu: {x}, {y}")
    case (x, y):
        print(f"Jiny bod: {x}, {y}")

Bod na ose y, y = 4.5


Podobně se dá rozebrat slovník - vzor sedí, když slovník obsahuje uvedené
klíče a hodnoty. Ostatní klíče nevadí.

In [59]:
prvek = {"typ": "nosnik", "rozpeti": 6.0, "vyska": 0.45}

match prvek:
    case {"typ": "nosnik", "rozpeti": L, "vyska": h}:
        print(f"Nosnik L = {L} m, h = {h} m, L/h = {L / h:.1f}")
    case {"typ": "deska", "tloustka": t}:
        print(f"Deska tloustky {t} m")
    case _:
        print("Neznamy prvek")

Nosnik L = 6.0 m, h = 0.45 m, L/h = 13.3


### Pozor: jméno ve vzoru není konstanta

Holé jméno ve vzoru **nic neporovnává** - zachytí jakoukoli hodnotu a uloží ji
pod tímto jménem. Následující kód proto vypíše "XD1", i když je očekávaná třída jiná.
Porovnávat se dá jen s literálem (`"XC1"`) nebo s tečkovaným jménem
(`Tridy.XC1`), proměnná na to nestačí.

In [60]:
OCEKAVANA = "XC1"
trida_prostredi = "XD1"

match trida_prostredi:
    case OCEKAVANA:          # neporovnava! zachyti cokoli
        print("Shoda s", OCEKAVANA)

print(OCEKAVANA)             # promenna byla prepsana na "XD1"

Shoda s XD1
XD1


## 10. Kontrola vstupu

Hodnoty od uživatele nebo z exportu je dobré před výpočtem zkontrolovat. Vstup
je vždycky text, takže se musíme zeptat na tři věci: jestli vůbec něco přišlo,
jestli se to dá převést na číslo a jestli je výsledek v rozumném rozsahu.

Zatím umíme jednu kontrolu provést jednou. Opakovat dotaz, dokud uživatel nezadá
platnou hodnotu, budeme umět v lekci 5. Výjimky (`try` / `except`), kterými se
nepovedený převod dá zachytit elegantněji, přijdou v lekci 9.

Pro vyzkoušení s `input()` nahraďte první řádek takto:

```python
zadani = input("Zadejte tloustku desky v m: ")
```

In [61]:
zadani = " 0,25 "     # misto input(); zkuste i "", "abc", "-0.3", "0.02", "2.5"

text = zadani.strip().replace(",", ".")
telo = text.removeprefix("-")    # znaminko resime zvlast, at ho pozname
je_cislo = telo.count(".") <= 1 and telo.replace(".", "").isdecimal() # max jedna tecka and vsechny znaky cislice
tloustka = float(text) if je_cislo else None

if not text:
    print("Nic nebylo zadano")
elif tloustka is None:
    print(f"'{zadani}' neni cislo")
elif tloustka <= 0:
    print(f"Tloustka musi byt kladna, zadano {tloustka} m")
elif not (0.05 <= tloustka <= 1.0):
    print(f"Tloustka {tloustka} m je mimo rozsah 0.05 az 1.0 m")
else:
    print(f"Tloustka desky {tloustka} m")

Tloustka desky 0.25 m


Jak to funguje:

* `removeprefix("-")` odloží znaménko stranou, takže záporné číslo projde
  kontrolou jako číslo a ohlásí se vlastní větví. Bez toho by `-0.3` skončilo
  mezi nečísly
* `count(".") <= 1` zařízne zápis se dvěma tečkami, `replace(".", "")` zbylou
  tečku odstraní a `isdecimal()` ověří, že zbyly jen číslice
* `float(text) if je_cislo else None` je podmíněný výraz z části 7. Převod se
  provede jen tehdy, když má smysl, a kód zůstane plochý - dál pokračuje
  obyčejná řada `elif` s `is None` z části 6
* na pořadí větví záleží (část 3): prázdný vstup první, rozsah až nakonec, kdy
  už víme, že `tloustka` je kladné číslo

Na vlastní kontrole je důležité jedno: musí kopírovat pravidla funkce `float()`
a každá mezera mezi nimi je chyba. Tahle verze odmítne `+0.25` i `1e-3`, přestože
`float()` je přijme. Pro kladnou tloušťku desky to nevadí, u obecného vstupu už
ano - proto se v Pythonu častěji převod rovnou zkusí a případný neúspěch se
zachytí.

## 11. Časté chyby

**Přiřazení místo porovnání.** `=` přiřazuje, `==` porovnává. V podmínce
skončí `=` chybou `SyntaxError`.

In [62]:
beton = "C25/30"
if beton = "C25/30":
    print("ano")

SyntaxError: invalid syntax. Maybe you meant '==' or ':=' instead of '='? (2525294987.py, line 2)

**Chybějící dvojtečka nebo odsazení.**

In [63]:
vyuziti = 0.8
if vyuziti <= 1.0:
print("Vyhovuje")

IndentationError: expected an indented block after 'if' statement on line 2 (2716556687.py, line 3)

**`or` s hodnotou místo porovnání.** Podmínka níže je vždy pravdivá: čte se jako
`(trida == "XC1") or ("XC2")` a neprázdný řetězec `"XC2"` je pravdivý.

In [64]:
trida = "XD3"

if trida == "XC1" or "XC2":           # spatne - vzdy True
    print("Spatne: vypise se pro kazdou tridu")

if trida == "XC1" or trida == "XC2":  # spravne
    print("Trida XC1 nebo XC2")

if trida in ("XC1", "XC2"):           # spravne a kratsi
    print("Trida XC1 nebo XC2")

Spatne: vypise se pro kazdou tridu


**Porovnání desetinných čísel pomocí `==`.** Z lekce 2 víme, že `0.1 + 0.2`
není přesně `0.3`. V podmínce použijte `math.isclose()`, nebo porovnávejte
nerovností s tolerancí.

In [65]:
import math

soucet_podilu = 0.6 + 0.3 + 0.1

if soucet_podilu == 1.0:
    print("Podily davaji 100 %")
else:
    print(f"Podily nedavaji 100 %? soucet = {soucet_podilu!r}")

if math.isclose(soucet_podilu, 1.0):
    print("S toleranci: podily davaji 100 %")

Podily nedavaji 100 %? soucet = 0.9999999999999999
S toleranci: podily davaji 100 %


## 12. Výraz a příkaz

Dva pojmy, které se budou hodit po zbytek kurzu:

* **výraz** (expression) - má hodnotu, dá se vyhodnotit a přiřadit proměnné
* **příkaz** (statement) - ovlivňuje běh (něco provede), hodnotu nemá

| Výraz | Příkaz |
|---|---|
| `25`, `delka * sirka` | `x = 5` |
| `0 <= x <= L` | `if`, `elif`, `else` |
| `float(text)`, `max(vymery)` | `for`, `while` |
| `"ano" if plati else "ne"` | `def`, `return`, `import` |

Jednoduchý test: **dá se to napsat za `x = ` nebo dovnitř `print()`?** Pokud ano,
je to výraz.

Operátor `:=` (tzv. walrus, od Pythonu 3.8) je **přiřazovací výraz**: přiřadí hodnotu a zároveň ji vrátí.

In [66]:
a = (b = 1)

SyntaxError: invalid syntax. Maybe you meant '==' or ':=' instead of '='? (3714954593.py, line 1)

In [67]:
a = (b := 1)
print(a)
print(b)

1
1


## Shrnutí

| Zápis | Kdy použít |
|---|---|
| `if` | provést blok jen při splnění podmínky |
| `if` / `else` | jedna ze dvou možností |
| `if` / `elif` / `else` | jedna z více možností, platí první splněná |
| `a if podminka else b` | volba jedné hodnoty na jednom řádku nebo ve vyrazu |
| `match` / `case` | jedna hodnota proti řadě konstant nebo tvarů |

Co si odnést:

* blok tvoří odsazení, za podmínkou je dvojtečka
* v řadě `elif` se provede jen první splněná větev - začněte nejpřísnější mezí
* `0`, `""`, `[]`, `{}` a `None` jsou nepravdivé; na chybějící hodnotu se ptejte
  `is None`
* `x == "A" or "B"` je vždy pravda, správně je `x in ("A", "B")`
* jméno ve vzoru `case` hodnotu zachytí, neporovná ji
* výraz má hodnotu a dá se přiřadit, příkaz ne; přiřazení `=` je příkaz

V lekci 5 přidáme cykly a projdeme kolekce prvek po prvku - každý prvek pak
budeme moci posoudit zvlášť.